# ワールドカップ2026 グループステージ 得点時刻の統計分析

有意水準 α = 0.05。H0-1（前半比率 p = 0.5、母比率の検定）を主要仮説とし、
H0-2（8区分の一様性、カイ二乗適合度検定）と H0-3（強豪国と非強豪国の終盤得点割合、
母比率の差の検定）を副次仮説とする。H0-1 と H0-3 は両側検定。
各検定は「棄却域」と「実現値」の比較で判定する。

In [ ]:
!pip install japanize-matplotlib -q
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import japanize_matplotlib
plt.rcParams['figure.dpi'] = 110

ALPHA  = 0.05
z_025  = stats.norm.ppf(1 - ALPHA/2)
crit_7 = stats.chi2.ppf(1 - ALPHA, 7)
crit_5 = stats.chi2.ppf(1 - ALPHA, 5)
print(f'z_0.025 = {z_025:.4f} (数表 1.960) , '
      f'chi2_0.05(7) = {crit_7:.4f} (数表 14.067) , '
      f'chi2_0.05(5) = {crit_5:.4f} (数表 11.070)')

## 1. データ読み込み・派生列・検証

`worddcup_scoredata.xlsx`（match_no / scoring_team / minute / added_time）を選ぶ。
アップロードが失敗する場合は左のファイルブラウザに直接置き、
`FN = 'worddcup_scoredata.xlsx'` に書き換える。

In [ ]:
from google.colab import files
up = files.upload()
FN = list(up.keys())[0]

In [ ]:
df = pd.read_excel(FN) if FN.endswith(('.xlsx','.xls')) else pd.read_csv(FN)
df['added_time'] = df['added_time'].fillna(0).astype(int)
n = len(df)

def make_bin(m, a):
    if a > 0:
        return '45+' if m == 45 else '90+'
    if m <= 15: return '01-15'
    if m <= 30: return '16-30'
    if m <= 45: return '31-45'
    if m <= 60: return '46-60'
    if m <= 75: return '61-75'
    return '76-90'

ORDER = ['01-15','16-30','31-45','45+','46-60','61-75','76-90','90+']
df['bin']  = [make_bin(m,a) for m,a in zip(df['minute'], df['added_time'])]
df['half'] = np.where(df['minute'] <= 45, 1, 2)
df['late'] = df['minute'] >= 68
df['t']    = df['minute'] + df['added_time']

print('minute 欠損 / 範囲外          :', df['minute'].isna().sum(),
      '/', ((df['minute'] < 1) | (df['minute'] > 90)).sum())
print('AT>0 かつ minute が45/90以外  :', ((df['added_time']>0) & (~df['minute'].isin([45,90]))).sum())
print('完全重複行                    :', df.duplicated(subset=list(df.columns[:4])).sum())
print('得点のあった試合数            :', df['match_no'].nunique(), '（全72試合中）')
print(f'n = {n} , 前半 {(df["half"]==1).sum()} / 後半 {(df["half"]==2).sum()}')

## 2. H0-1　前半の得点割合は 0.5 か

In [ ]:
x, p0 = int((df['half'] == 1).sum()), 0.5
ph  = x / n
se0 = np.sqrt(p0*(1-p0)/n)
se1 = np.sqrt(ph*(1-ph)/n)
z1  = (ph - p0) / se0

print(f'適用条件 n*p0 = {n*p0:.1f} , n*(1-p0) = {n*(1-p0):.1f}  → ともに5以上')
print(f'棄却域 |z| ≥ {z_025:.3f}')
print(f'実現値 p̂ = {x}/{n} = {ph:.4f} , z = ({ph:.4f} - {p0}) / {se0:.5f} = {z1:.4f}')
print('→', '棄却する' if abs(z1) >= z_025 else '棄却されない')
print(f'p の95%信頼区間 : [{ph-z_025*se1:.4f}, {ph+z_025*se1:.4f}]')

In [ ]:
print(f'検出力  1-β = P(|Z| ≥ z_0.025 | p = p1)   （n = {n}）')
for p1 in [0.48, 0.46, 0.44, round(ph,3), 0.42, 0.40]:
    sa = np.sqrt(p1*(1-p1)/n)
    print(f'  p1 = {p1:.3f} → {stats.norm.cdf((abs(p1-p0) - z_025*se0)/sa):.3f}')

for nn in range(100, 5000, 5):
    s0, s1 = np.sqrt(p0*(1-p0)/nn), np.sqrt(ph*(1-ph)/nn)
    if stats.norm.cdf((abs(ph-p0) - z_025*s0)/s1) >= 0.80:
        print(f'p̂ = {ph:.3f} を検出力0.8で検出するのに必要な n = {nn}')
        break

## 3. H0-2　8区分の一様性

In [ ]:
obs = df['bin'].value_counts().reindex(ORDER, fill_value=0).values
E   = n / 8
chi2_1 = ((obs - E)**2 / E).sum()

print(f'期待度数 Ei = {n}/8 = {E:.3f}  → すべて5以上')
print(f'棄却域 χ² ≥ {crit_7:.3f} ,  実現値 χ² = {chi2_1:.4f}')
print('→', '棄却する' if chi2_1 >= crit_7 else '棄却されない')
print(f'\n{"区分":>7}{"O":>6}{"E":>9}{"(O-E)²/E":>11}{"寄与%":>8}{"標準化残差":>12}')
for k, o in zip(ORDER, obs):
    c = (o-E)**2/E
    print(f'{k:>7}{o:>6}{E:>9.2f}{c:>11.4f}{100*c/chi2_1:>8.1f}{(o-E)/np.sqrt(E):>12.2f}')

In [ ]:
# 感度分析：期待確率を区分長に比例させる
for a1, a2 in [(2,5),(3,6),(4,8),(5,10)]:
    L  = np.array([15,15,15,a1,15,15,15,a2], float)
    Ei = L/L.sum()*n
    c  = ((obs - Ei)**2/Ei).sum()
    print(f'前半AT={a1}分, 後半AT={a2}分 → χ² = {c:6.3f} → '
          f'{"棄却する" if c >= crit_7 else "棄却されない"}')

# ATを直前区分に合算した6区分
B6 = ['1-15','16-30','31-45','46-60','61-75','76-90']
o6 = pd.cut(df['minute'], [0,15,30,45,60,75,90], labels=B6).value_counts().reindex(B6).values
c6 = ((o6 - n/6)**2/(n/6)).sum()
print(f'\n6区分 {dict(zip(B6, o6))} 期待度数 {n/6:.2f}')
print(f'χ² = {c6:.4f} , 棄却域 χ² ≥ {crit_5:.3f} →',
      '棄却する' if c6 >= crit_5 else '棄却されない')

## 4. H0-3　強豪国と非強豪国の終盤得点割合

In [ ]:
RANK = {
 "Mexico":14,"South Africa":60,"South Korea":25,"Czech Republic":39,
 "Canada":30,"Bosnia and Herzegovina":64,"Qatar":57,"Switzerland":19,
 "Brazil":6,"Morocco":7,"Haiti":83,"Scotland":42,
 "USA":17,"Paraguay":40,"Australia":27,"Turkey":22,
 "Germany":10,"Curacao":82,"Ivory Coast":33,"Ecuador":23,
 "Netherlands":8,"Japan":18,"Sweden":38,"Tunisia":46,
 "Belgium":9,"Egypt":29,"Iran":21,"New Zealand":85,
 "Spain":2,"Cape Verde":67,"Saudi Arabia":61,"Uruguay":16,
 "France":3,"Senegal":15,"Iraq":56,"Norway":31,
 "Argentina":1,"Algeria":28,"Austria":24,"Jordan":63,
 "Portugal":5,"DR Congo":45,"Uzbekistan":50,"Colombia":13,
 "England":4,"Croatia":11,"Ghana":73,"Panama":34,
}
assert not (set(df['scoring_team']) - set(RANK)), '未対応のチーム名あり'
df['rank']   = df['scoring_team'].map(RANK)
df['strong'] = df['rank'] <= 16
print('強豪国（16位以内）:', sum(v <= 16 for v in RANK.values()), 'か国')

In [ ]:
x1, n1 = int((df['strong'] & df['late']).sum()),  int(df['strong'].sum())
x2, n2 = int((~df['strong'] & df['late']).sum()), int((~df['strong']).sum())
p1h, p2h = x1/n1, x2/n2
pbar = (x1+x2)/(n1+n2)
se   = np.sqrt(pbar*(1-pbar)*(1/n1 + 1/n2))
z3   = (p1h - p2h)/se

print(f'適用条件 n1p̄={n1*pbar:.1f}, n1(1-p̄)={n1*(1-pbar):.1f}, '
      f'n2p̄={n2*pbar:.1f}, n2(1-p̄)={n2*(1-pbar):.1f} → すべて5以上')
print(f'棄却域 |z| ≥ {z_025:.3f}')
print(f'実現値 p̂1 = {x1}/{n1} = {p1h:.4f} , p̂2 = {x2}/{n2} = {p2h:.4f} , p̄ = {pbar:.4f}')
print(f'       z = ({p1h:.4f} - {p2h:.4f}) / {se:.5f} = {z3:.4f}')
print('→', '棄却する' if abs(z3) >= z_025 else '棄却されない')

sed, d = np.sqrt(p1h*(1-p1h)/n1 + p2h*(1-p2h)/n2), p1h - p2h
print(f'比率の差 {d:+.4f} , 95%信頼区間 [{d-z_025*sed:+.4f}, {d+z_025*sed:+.4f}]')

In [ ]:
# 感度分析：強豪国と終盤の定義を変える
for th, lab in [(16,'16位以内(事前登録)'), (17,'出場国中の上位16')]:
    for lt in [68, 75]:
        s, l = df['rank'] <= th, df['minute'] >= lt
        a, b = int((s&l).sum()), int(s.sum())
        c, e = int((~s&l).sum()), int((~s).sum())
        P = (a+c)/(b+e); S = np.sqrt(P*(1-P)*(1/b+1/e)); Z = (a/b - c/e)/S
        print(f'{lab}, {lt}分以降 → p̂1={a/b:.3f}, p̂2={c/e:.3f}, z={Z:+.4f} → '
              f'{"棄却する" if abs(Z) >= z_025 else "棄却されない"}')

## 5. 補足分析　FIFAランキングと平均得点時刻（検定は行わない）

In [ ]:
g = df.groupby('scoring_team').agg(n=('t','size'), mean_t=('t','mean')).reset_index()
g['rank'] = g['scoring_team'].map(RANK)
g3 = g[g['n'] >= 3]                       # 得点3点以上に限定（事前確定）
r  = np.corrcoef(g3['rank'], g3['mean_t'])[0,1]
print(f'対象 {len(g3)} チーム（全{len(g)}チーム中） , r = {r:.4f} , r² = {r**2:.4f}')

## 6. 作図（LaTeX用にPDF保存）

貼り付け前に `extractbb ファイル名.pdf` を一度実行する。

In [ ]:
fig, ax = plt.subplots(figsize=(7.5,4.2))
ax.bar(range(8), obs, color=['#4a7ba7' if o>=E else '#c4703d' for o in obs],
       edgecolor='black', lw=0.6, zorder=3)
ax.axhline(E, color='crimson', ls='--', lw=1.4, label=f'一様仮説の期待度数 ({E:.1f})')
for i,o in enumerate(obs): ax.text(i, o+0.9, str(o), ha='center', fontsize=9)
ax.set_xticks(range(8)); ax.set_xticklabels(ORDER, fontsize=9)
ax.set_xlabel('時間区分（分）'); ax.set_ylabel('得点数'); ax.set_title('時間区分別の得点数')
ax.legend(); ax.grid(axis='y', alpha=.3, zorder=0)
fig.tight_layout(); fig.savefig('fig1_bins.pdf'); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(8.5,3.9))
h = [(df['half']==1).sum(), (df['half']==2).sum()]
axes[0].bar(['前半','後半'], h, color=['#7a9cc0','#c4703d'], edgecolor='black', lw=.6, zorder=3)
axes[0].axhline(n/2, color='crimson', ls='--', lw=1.4, label=f'期待度数 ({n/2:.1f})')
for i,v in enumerate(h): axes[0].text(i, v+2, str(v), ha='center')
axes[0].set_ylabel('得点数'); axes[0].legend(); axes[0].grid(axis='y', alpha=.3, zorder=0)
axes[1].errorbar([0],[ph], yerr=[z_025*se1], fmt='o', capsize=7, markersize=9, lw=2, color='#4a7ba7')
axes[1].axhline(0.5, color='crimson', ls='--', lw=1.4, label='帰無仮説 p = 0.5')
axes[1].set_xlim(-.6,.6); axes[1].set_xticks([]); axes[1].set_ylabel('前半得点の割合')
axes[1].legend(); axes[1].grid(axis='y', alpha=.3)
fig.tight_layout(); fig.savefig('fig2_half.pdf'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,3.8))
ax.hist(df['t'], bins=np.arange(0,105,5), color='#6b8fb5', edgecolor='black', lw=.6, zorder=3)
ax.axvline(45, color='gray', ls=':', lw=1.5); ax.axvline(90, color='gray', ls=':', lw=1.5)
ax.set_xlabel('得点時刻（分、アディショナルタイム加算）'); ax.set_ylabel('得点数')
ax.grid(axis='y', alpha=.3, zorder=0)
fig.tight_layout(); fig.savefig('fig3_hist.pdf'); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(9,3.9))
errs = [z_025*np.sqrt(p*(1-p)/nn) for p,nn in [(p1h,n1),(p2h,n2)]]
axes[0].bar(['強豪国\n(16位以内)','非強豪国\n(17位以下)'], [p1h,p2h], yerr=errs, capsize=8,
            color=['#4a7ba7','#c4703d'], edgecolor='black', lw=.6, zorder=3)
axes[0].set_ylabel('68分以降の得点の割合'); axes[0].set_ylim(0,.52)
axes[0].grid(axis='y', alpha=.3, zorder=0)
axes[1].scatter(g3['rank'], g3['mean_t'], s=g3['n']*7, alpha=.65,
                color='#4a7ba7', edgecolor='black', lw=.5, zorder=3)
xs = np.linspace(g3['rank'].min(), g3['rank'].max(), 50)
axes[1].plot(xs, np.polyval(np.polyfit(g3['rank'], g3['mean_t'], 1), xs),
             color='crimson', ls='--', lw=1.4)
axes[1].set_xlabel('FIFAランキング'); axes[1].set_ylabel('平均得点時刻（分）')
axes[1].set_title(f'r = {r:.3f}（検定は行わない）', fontsize=10); axes[1].grid(alpha=.3, zorder=0)
fig.tight_layout(); fig.savefig('fig4_strength.pdf'); plt.show()

In [ ]:
from google.colab import files
for f in ['fig1_bins.pdf','fig2_half.pdf','fig3_hist.pdf','fig4_strength.pdf']:
    files.download(f)